## IMPORTAR PAQUETES

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
import janitor
import os
from pathlib import Path
import openpyxl
import sqlalchemy as sa

# Desactivar notación científica
pd.set_option('display.float_format', lambda x: '%.3f' % x)
np.set_printoptions(suppress=True)

# Cargar variables de entorno
load_dotenv()

print("✅ Librerías importadas correctamente")

## IMPORTAR LOS DATOS

In [2]:
num = pd.read_pickle('../02_datos/03_Entrenamiento/num_resultado_calidad.pkl')
cat = pd.read_pickle('../02_datos/03_Entrenamiento/cat_resultado_calidad.pkl')

# Nuevas Variables

- Las identificadas en EDA
- Lags
- Ventanas móviles

In [8]:
df = pd.concat([cat, num], axis = 1 )
df 

,store_id,item_id,d,year,month,wday,weekday,event_name_1,event_type_1,ventas,sell_price,wm_yr_wk
date,,,,,,,,,,,,
2013-01-01,CA_3,FOODS_3_090,d_704,2013,1,4,Tuesday,NewYear,National,0,1.250,11249
2013-01-01,CA_3,FOODS_3_120,d_704,2013,1,4,Tuesday,NewYear,National,33,1.250,11249
2013-01-01,CA_3,FOODS_3_202,d_704,2013,1,4,Tuesday,NewYear,National,0,4.980,11249
2013-01-01,CA_3,FOODS_3_252,d_704,2013,1,4,Tuesday,NewYear,National,0,4.980,11249
2013-01-01,CA_3,FOODS_3_288,d_704,2013,1,4,Tuesday,NewYear,National,20,4.280,11249
...,...,...,...,...,...,...,...,...,...,...,...,...
2015-11-30,CA_4,FOODS_3_329,d_1767,2015,11,3,Monday,Sin evento,Sin evento,9,1.680,11544
2015-11-30,CA_4,FOODS_3_555,d_1767,2015,11,3,Monday,Sin evento,Sin evento,26,2.480,11544
2015-11-30,CA_4,FOODS_3_586,d_1767,2015,11,3,Monday,Sin evento,Sin evento,13,2.480,11544


## Variable demanda intermitente

Esta variable va a identificar cuantos días seguidos han transcurrido con ventas cero

In [17]:
def rotura_stock(ventas, n = 5):
    cero_ventas = pd.Series(np.where(ventas == 0, 1, 0))
    num_ceros = cero_ventas.rolling(n).sum()
    rotura_stock = np.where(num_ceros == n, 1, 0)
    return (rotura_stock)

In [13]:
df = df.sort_values(by = ['store_id','item_id','date'])

In [18]:
df['rotura_stock_3'] = df.groupby(['store_id', 'item_id']).ventas.transform(lambda x: rotura_stock(x, 3)).values

In [20]:
df['rotura_stock_7'] = df.groupby(['store_id', 'item_id']).ventas.transform(lambda x: rotura_stock(x, 7)).values

In [23]:
df['rotura_stock_15'] = df.groupby(['store_id', 'item_id']).ventas.transform(lambda x: rotura_stock(x, 15)).values

## Variables de Lag

- ventas: lags de 15 días
- sell_price: lags de 7 días
- rotura_stock: lag de un día

In [25]:
def crear_lags(df, variable, num_lags = 7):

    # Crear el objeto dataframe
    lags = pd.DataFrame()

    # Crea todos los lags 
    for cada in range(1, num_lags + 1):
        lags[variable + '_lag_' + str(cada)] = df[variable].shift(cada)

    # Devuelve el dataframe con los lags    
    return(lags)

In [35]:
lags_sell_price_df = (df.groupby(['store_id', 'item_id'])
                        .apply(lambda x: crear_lags(df = x, variable = 'sell_price', num_lags= 7))
                        .reset_index()
                        .set_index('date'))

In [36]:
lags_rotura_stock_3_df = (df.groupby(['store_id', 'item_id'])
                            .apply(lambda x: crear_lags(df = x, variable = 'rotura_stock_3', num_lags= 1))
                            .reset_index()
                            .set_index('date'))

In [37]:
lags_rotura_stock_7_df = (df.groupby(['store_id', 'item_id'])
                                .apply(lambda x : crear_lags(df = x, variable= 'rotura_stock_7', num_lags= 1))
                                .reset_index()
                                .set_index('date'))

In [38]:
lags_rotura_stock_15_df = (df.groupby(['store_id', 'item_id'])
                                .apply(lambda x : crear_lags(df = x, variable= 'rotura_stock_15', num_lags= 1))
                                .reset_index()
                                .set_index('date'))

In [39]:
lags_ventas_df = (df.groupby(['store_id', 'item_id'])
                        .apply(lambda x : crear_lags(df= x, variable='ventas', num_lags= 15))
                        .reset_index()
                        .set_index('date'))

In [52]:
lags_rotura_stock_7_df

,store_id,item_id,rotura_stock_7_lag_1
date,,,
2013-01-01,CA_3,FOODS_3_090,NaN
2013-01-02,CA_3,FOODS_3_090,0.000
2013-01-03,CA_3,FOODS_3_090,0.000
2013-01-04,CA_3,FOODS_3_090,0.000
2013-01-05,CA_3,FOODS_3_090,0.000
...,...,...,...
2015-11-26,CA_4,FOODS_3_714,0.000
2015-11-27,CA_4,FOODS_3_714,0.000
2015-11-28,CA_4,FOODS_3_714,0.000


## Variables de ventanas móviles

- Mínimo móvil
- Media móvil
- Máximo móvil

Rangos de 15 días

In [46]:
def min_movil(df, variable, num_periodos= 7):

    minm = pd.DataFrame()

    for cada in range(2, num_periodos+1):
        minm[variable + '_minm_' + str(cada)] = df[variable].shift(1).rolling(cada).min()

    return(minm)

In [59]:
def media_movil(df, variable, num_periodos= 7):

    mm = pd.DataFrame()

    for cada in range(2, num_periodos+1):
       mm[variable + '_mm_' + str(cada)] = df[variable].shift(1).rolling(cada).mean()

    return(mm)

In [49]:
def max_movil(df, variable, num_periodos= 7):

    maxm = pd.DataFrame()

    for cada in range(2, num_periodos+1):
        maxm[variable + '_maxm_' + str(cada)] = df[variable].shift(1).rolling(cada).max()

    return(maxm)

In [57]:
min_movil_df = (df.groupby(['store_id', 'item_id'])
                    .apply(lambda x : min_movil(df = x, variable= 'ventas', num_periodos= 15))
                    .reset_index()
                    .set_index('date'))

In [60]:
media_movil_df = (df.groupby(['store_id', 'item_id'])
                    .apply(lambda x : media_movil(df = x, variable= 'ventas', num_periodos= 15))
                    .reset_index()
                    .set_index('date'))

In [61]:
max_movil_df = (df.groupby(['store_id', 'item_id'])
                    .apply(lambda x : max_movil(df = x, variable= 'ventas', num_periodos= 15))
                    .reset_index()
                    .set_index('date'))

# Preparar los Datasets

## Unir todos los dataframes generados

In [63]:
df_unido = pd.concat([df,
             lags_sell_price_df,
             lags_rotura_stock_3_df,
             lags_rotura_stock_7_df,
             lags_rotura_stock_15_df,
             lags_ventas_df,
             min_movil_df,
             media_movil_df,
             max_movil_df], axis = 1 )

# Eliminar columnas duplicadas
df_unido = df_unido.loc[:,~df_unido.columns.duplicated()]
df_unido

,store_id,item_id,d,year,month,wday,weekday,event_name_1,event_type_1,ventas,...,ventas_maxm_6,ventas_maxm_7,ventas_maxm_8,ventas_maxm_9,ventas_maxm_10,ventas_maxm_11,ventas_maxm_12,ventas_maxm_13,ventas_maxm_14,ventas_maxm_15
date,,,,,,,,,,,,,,,,,,,,,
2013-01-01,CA_3,FOODS_3_090,d_704,2013,1,4,Tuesday,NewYear,National,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2013-01-02,CA_3,FOODS_3_090,d_705,2013,1,5,Wednesday,Sin evento,Sin evento,224,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2013-01-03,CA_3,FOODS_3_090,d_706,2013,1,6,Thursday,Sin evento,Sin evento,241,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2013-01-04,CA_3,FOODS_3_090,d_707,2013,1,7,Friday,Sin evento,Sin evento,232,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2013-01-05,CA_3,FOODS_3_090,d_708,2013,1,1,Saturday,Sin evento,Sin evento,301,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2015-11-26,CA_4,FOODS_3_714,d_1763,2015,11,6,Thursday,Thanksgiving,National,22,...,15.000,15.000,15.000,15.000,15.000,17.000,17.000,17.000,17.000,17.000
2015-11-27,CA_4,FOODS_3_714,d_1764,2015,11,7,Friday,Sin evento,Sin evento,10,...,22.000,22.000,22.000,22.000,22.000,22.000,22.000,22.000,22.000,22.000
2015-11-28,CA_4,FOODS_3_714,d_1765,2015,11,1,Saturday,Sin evento,Sin evento,17,...,22.000,22.000,22.000,22.000,22.000,22.000,22.000,22.000,22.000,22.000


## Eliminar los nulos generados por las nuevas variables

In [64]:
df_unido.dropna(inplace= True)

## Eliminar las variables que no vamos a necesitar para modelizar

In [65]:
a_eliminar = ['d', 'wm_yr_wk', 'sell_price', 'rotura_stock_3', 'rotura_stock_7', 'rotura_stock_15']

In [66]:
df_unido.drop(columns=a_eliminar, inplace=True)

## Identificar target

In [67]:
target = df_unido.ventas

## Separar num y cat

In [69]:
cat = df_unido.select_dtypes(include=['object', 'string'])

In [70]:
num = df_unido.select_dtypes(exclude=['object', 'string'])

# TRANSFORMACIÓN DE CATEGÓRICAS

## One Hot Encoding

### Variables a aplicar OHT

In [81]:
var_ohe = ['year',
           'month',
           'wday',
           'weekday',
           'event_type_1',
           'event_name_1'
           ]

### Instanciar

In [79]:
from sklearn.preprocessing import OneHotEncoder
ohe = OneHotEncoder(sparse_output = False, handle_unknown='ignore')

### Entrenar y aplicar

In [82]:
cat_ohe = ohe.fit_transform(cat[var_ohe])

### Guardar como dataframe

In [83]:
cat_ohe = pd.DataFrame(cat_ohe, columns = ohe.get_feature_names_out())

## Target Encoding

### Variables a aplicar TE

In [84]:
var_te = [
    'year',
    'month',
    'wday',
    'weekday',
    'event_name_1',
    'event_type_1'
]

### Instanciar

In [85]:
from category_encoders import TargetEncoder
te = TargetEncoder(min_samples_leaf=100, return_df = False)

### Entrenar y aplicar

In [87]:
cat_te = te.fit_transform(cat[var_te], y = target)

c:\Users\phdel\miniconda3\envs\ForecastingRetail\Lib\contextlib.py:141: Pandas4Warning: 'future.no_silent_downcasting' is deprecated, please refrain from using it.
  return next(self.gen)
c:\Users\phdel\miniconda3\envs\ForecastingRetail\Lib\contextlib.py:141: Pandas4Warning: 'future.no_silent_downcasting' is deprecated, please refrain from using it.
  return next(self.gen)
c:\Users\phdel\miniconda3\envs\ForecastingRetail\Lib\contextlib.py:148: Pandas4Warning: 'future.no_silent_downcasting' is deprecated, please refrain from using it.
  next(self.gen)
c:\Users\phdel\miniconda3\envs\ForecastingRetail\Lib\contextlib.py:141: Pandas4Warning: 'future.no_silent_downcasting' is deprecated, please refrain from using it.
  return next(self.gen)
c:\Users\phdel\miniconda3\envs\ForecastingRetail\Lib\contextlib.py:141: Pandas4Warning: 'future.no_silent_downcasting' is deprecated, please refrain from using it.
  return next(self.gen)
c:\Users\phdel\miniconda3\envs\ForecastingRetail\Lib\contextlib.py:

### Guardar como dataframe

In [88]:
# Añadir sufijos a los nombres
nombres_te = [variable + '_te' for variable in var_te]

# Guardar como dataframe
cat_te = pd.DataFrame(cat_te, columns = nombres_te)

# UNIFICAR DATASETS TRANSFORMADOS

## Meter en una lista todos los dataframes creados

Rescatar de df_unido las variables de segmentación

In [89]:
de_df_unido = df_unido[['store_id', 'item_id']].reset_index()

de_df_unido.head(2)

,date,store_id,item_id
0,2013-01-16,CA_3,FOODS_3_090
1,2013-01-17,CA_3,FOODS_3_090


## Unir todos los dataframes

In [92]:
dataframes = [de_df_unido, cat_ohe, cat_te, num.reset_index(drop=True)]

In [93]:
df_tablon = pd.concat(dataframes, axis=1)

df_tablon

,date,store_id,item_id,year_2013,year_2014,year_2015,month_1,month_2,month_3,month_4,...,ventas_maxm_6,ventas_maxm_7,ventas_maxm_8,ventas_maxm_9,ventas_maxm_10,ventas_maxm_11,ventas_maxm_12,ventas_maxm_13,ventas_maxm_14,ventas_maxm_15
0,2013-01-16,CA_3,FOODS_3_090,1.000,0.000,0.000,1.000,0.000,0.000,0.000,...,351.000,351.000,351.000,351.000,351.000,351.000,351.000,351.000,351.000,351.000
1,2013-01-17,CA_3,FOODS_3_090,1.000,0.000,0.000,1.000,0.000,0.000,0.000,...,351.000,351.000,351.000,351.000,351.000,351.000,351.000,351.000,351.000,351.000
2,2013-01-18,CA_3,FOODS_3_090,1.000,0.000,0.000,1.000,0.000,0.000,0.000,...,351.000,351.000,351.000,351.000,351.000,351.000,351.000,351.000,351.000,351.000
3,2013-01-19,CA_3,FOODS_3_090,1.000,0.000,0.000,1.000,0.000,0.000,0.000,...,281.000,351.000,351.000,351.000,351.000,351.000,351.000,351.000,351.000,351.000
4,2013-01-20,CA_3,FOODS_3_090,1.000,0.000,0.000,1.000,0.000,0.000,0.000,...,362.000,362.000,362.000,362.000,362.000,362.000,362.000,362.000,362.000,362.000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
20975,2015-11-26,CA_4,FOODS_3_714,0.000,0.000,1.000,0.000,0.000,0.000,0.000,...,15.000,15.000,15.000,15.000,15.000,17.000,17.000,17.000,17.000,17.000
20976,2015-11-27,CA_4,FOODS_3_714,0.000,0.000,1.000,0.000,0.000,0.000,0.000,...,22.000,22.000,22.000,22.000,22.000,22.000,22.000,22.000,22.000,22.000
20977,2015-11-28,CA_4,FOODS_3_714,0.000,0.000,1.000,0.000,0.000,0.000,0.000,...,22.000,22.000,22.000,22.000,22.000,22.000,22.000,22.000,22.000,22.000
20978,2015-11-29,CA_4,FOODS_3_714,0.000,0.000,1.000,0.000,0.000,0.000,0.000,...,22.000,22.000,22.000,22.000,22.000,22.000,22.000,22.000,22.000,22.000


# GUARDAR DATASET TRAS TRANSFORMACIÓN DE DATOS

En formato pickle para no perder las modificaciones de metadatos

In [94]:
# Guardar los archivos
df_tablon.to_pickle('../02_datos/03_Entrenamiento/df_tablon_transformado.pkl')